# GyrazeInterface HDF5 File Structure

#### Overview
The `GyrazeInterface` class converts Gkeyll simulation data into the format required by the GYRAZE code, storing multiple datasets in a single HDF5 file.

```text
gkeyll_gyraze_inputs.h5
├── attributes:
│   ├── description: "Gyraze input data files from Gkeyll simulation"
│   └── nsample: <number of datasets>
│
├── Group: "000001"
│   ├── datasets:
│   │   ├── Fe_mpe_args.txt    # Electron velocity/mu grid parameters
│   │   ├── Fe_mpe.txt         # Electron distribution function at magnetic presheath entrance
│   │   ├── Fi_mpe_args.txt    # Ion velocity/mu grid parameters 
│   │   ├── Fi_mpe.txt         # Ion distribution function at magnetic presheath entrance
│   │   │── input_physparams.txt # Physical parameters for GYRAZE
│   │   └── input_numparams.txt  # Numerical parameters for GYRAZE
│   │
│   └── attributes:
│       ├── x0, y0, z0: spatial coordinates
│       ├── t0: time
│       ├── tf: time frame index
│       ├── alphadeg: field line angle
│       ├── B0, phi0: magnetic field and potential
│       ├── ne0, ni0: electron and ion densities
│       ├── Te0, Ti0: electron and ion temperatures
│       ├── gamma0: rhoe_lambdaD parameter
│       ├── nioverne, TioverTe: density and temperature ratios
│       ├── mioverme, mi, me, e: mass and charge constants
│       └── simprefix: simulation path
│
├── Group: "000002"
│   └── ... (same structure)
│
└── ... (additional groups for each sampled point)
```

#### Usage
You will need to first load your Gkeyll simulation with the `pygkyl` library and then create an instance of `GyrazeInterface`. Below is an example workflow

#### NOTES:
Normalize distf so that the integral over normalized velocity is 1.0
Each species gets their own normalization, e.g. int f_i dv = ni becomes int hatf_i dvhat_i = 1.0
Grid normalization: sqrt(2) in the vth?
Also set ni/ne = 1.0 so rescale so that it is true for the distribution function.

- It is expected that the new sheath BC (with mu dependence on vcut) will affect conducting sheath BC more than insulating sheath BC. 
- Allowing higher mu values to escape would increase the heat flux because we will lose ions with larger larmor radii that will not see the potential drop.
- Interesting to see also in the radial 
- Conducting sheath in Gkeyll : P.50 of Eric Shi thesis 2017.
- It would be great to have an analytical fit as the output of the surrogate instead of a table, e.g.:
    - vcut(mu) = a + b*mu + c*mu^2 + ... with coefficients a,b,c,... provided by the surrogate model.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import os
import pygkyl

# High resolution TCV simulation
simdir = '/Users/ahoffman/personal_gkyl_scripts/sim_data/tcv_pt_hd_3x2v_80/'
fileprefix = 'rt_gk_tcv_nt_iwl_adapt_src_3x2v_p1'
outfilename = 'gyraze_input_data_tcv_pt_hd_phi_lt_6.h5'
config = 'tcv_nt'

# DIII-D simulation with fluid neutrals
# simdir = '/Users/ahoffman/personal_gkyl_scripts/sim_data/3x2v/d3d-nt-neut-vis/'
# fileprefix = 'gk_d3d_iwl_adapt_source_3x2v_p1'
# outfilename = 'gyraze_input_data_d3d_nt_neut_vis.h5'
# config = 'd3d_pt'

# Coarse resolution TCV simulation (given to Patrick)
# simdir = '/Users/ahoffman/personal_gkyl_scripts/sim_data/3x2v/gk_tcv_adapt_src'
# fileprefix = 'rt_gk_tcv_nt_iwl_3x2v_p1'
# outfilename = 'gyraze_input_data_tcv_nt_coarse.h5'
# config = 'tcv_pt'

simulation = pygkyl.load_sim_config(configName=config, simDir=simdir, filePrefix=fileprefix)
# simulation.normalization.set('fluid velocities','thermal velocity') # fluid velocity moments are normalized by the thermal velocity
simulation.normalization.set('temperatures','eV') # temperatures in electron Volt
simulation.normalization.set('pressures','Pa') # pressures in Pascal
simulation.normalization.set('energies','MJ') # energies in mega Joules
simulation.normalization.set('gradients','major radius') # gradients are normalized by the major radius
simulation.normalization.change('mue', 1.0, 0.0, r'$\mu_e^*$', '')
simulation.normalization.change('vpare', 1.0, 0.0, r'$v_e^*$', '')
simulation.normalization.change('x', 1.0, 0.0, r'$x^*$', 'm')
sim_frames = simulation.available_frames['field'] # you can check the available frames for each data type like ion_M0, ion_BiMaxwellian, etc.)
print("%g time frames available (%g to %g)"%(len(sim_frames),sim_frames[0],sim_frames[-1]))

In [ ]:
simulation.plot_1D('y', cutCoords=[1.1, -1], frameIdx=sim_frames[-1], fieldName=['Tpare','Tperpe'])

In [ ]:
simulation.plot_2D('xy', cutCoords=[0], frameIdx=simulation.available_frames['field'][-1], 
                   fieldName=['ne','Te','Ti','phi','gamma_gyraze','phi_gyraze'], 
                   xlim=[simulation.geom_param.x_LCFS,simulation.geom_param.x_LCFS+simulation.geom_param.x_out], cmap='viridis',
                   clim = [[1e17,5e18],[10,50],[10,300],[0,500],[0,0.5],[0,5]])

Normalize distf so that the integral over normalized velocity is 1.0
Each species gets their own normalization, e.g. int f_i dv = ni becomes int hatf_i dvhat_i = 1.0
Grid normalization: sqrt(2) in the vth?
Also set ni/ne = 1.0 so rescale so that it is true for the distribution function.

- It is expected that the new sheath BC (with mu dependence on vcut) will affect conducting sheath BC more than insulating sheath BC. 
- Allowing higher mu values to escape would increase the heat flux because we will lose ions with larger larmor radii that will not see the potential drop.
- Interesting to see also in the radial 
- Conducting sheath in Gkeyll : P.50 of Eric Shi thesis 2017.

In [ ]:
gyraze = pygkyl.GyrazeInterface(simulation, number_datasets=True, outfilename=outfilename,
                                no_distf=True)
gyraze.generate(
    time_frames = sim_frames[:], 
    xmin = simulation.geom_param.x_LCFS + 0.01,
    xmax = simulation.geom_param.x_LCFS + simulation.geom_param.x_out - 0.01,
    Nxsample = 32,
    Nysample = 32,
    zplane = 'both',
    filter_negativity = True,
    verbose = False,
    no_distf=True,
    nsmooth_distf=1,
    vpos_fe=True,
    vpos_fi=True,
    int_fact_distf=1,
    spar_max_e=None,
    sperp_max_e=None,
    phase_space_norm='unnormalized',
    dens_pol=5e19,
    lim_dict = {
        # 'phi_norm': {'min': 3.0},
        'nioverne': {'max': 3.0},
        # 'gamma': {'min': 0.74, 'max': 4.0},
    }
)
# gyraze.load_h5_data('gyraze_input_data_tcv_pt_hd.h5')
# gyraze.plot_distf(idx=0, log_scale=False)
# gyraze.plot_distf(idx=0, log_scale=True)

# Export the data contained in the h5 file at a given index as individual files for use in Gyraze
sidx = 1
# gyraze.load_h5_data('gyraze_input_data_tcv_PT_hd.h5')
gyraze.load_h5_data(outfilename)
gyraze.extract_dataset_as_files(group_name=f'{sidx:06d}',output_dir='gyraze_data')

In [ ]:
# Visualize the physical parameters that will be transferred to Gyraze
gyraze.plot_data(alpha=0.1, figname=outfilename.replace('.h5','.png'))

In [ ]:
fxy = lambda x: -0.05*x + 7
gyraze.plot_attribute_scatter(attr_x='gamma0', attr_y='phi_norm0', color_by='x0', fxy=None)

In [ ]:
pygkyl.plot_utils.plot_2D_cut(simulation=simulation, 
                              fieldnames=['Bmag','phi','ne','ni','Te','Ti'],
                              cut_dir='xy',
                              cut_coord=0,
                              time_frame=sim_frames[-1],
                              clim=[[1.8,2.3],[0,300],[1e17,3e18],[1e17,5e18],[0,100],[0,300]],
                              xlim =[0.04,0.12]
)
pygkyl.plot_utils.plot_2D_cut(simulation=simulation, 
                              fieldnames=['rhoe','lambdaD','rhoe_lambdaD'],
                              cut_dir='xy',
                              cut_coord=0,
                              time_frame=sim_frames[-1],
                              clim=[[0.01,0.05],[0.001,0.02],[0.74,4]],
                              xlim =[0.04,0.12]
)

In [ ]:
pygkyl.plot_utils.plot_2D_cut(simulation=simulation, 
                              fieldnames=['fi','fe'],
                              cut_dir='vparmu',
                              time_frame=sim_frames[-1],
                              cut_coord=[0.10,0.0,0])

In [ ]:
pygkyl.plot_utils.plot_1D(simulation=simulation, 
                              fieldnames=['fi','fe'],
                              cdirection='vpar',
                              time_frames=sim_frames[-1],
                              ccoords=[0.10,0.0,0,0.0])

In [ ]:
pygkyl.plot_utils.plot_DG_representation(simulation,
                                         'fi',
                                         sim_frames[-1],
                                         cutdir='vpar',
                                         cutcoord=[0.10,0.0,0,0])

In [ ]:
# We can also load previously generated data
filename = '/Users/ahoffman/personal_gkyl_scripts/gkyl_gyrz.ignore/gyraze_input_data_tcv_PT_hd.h5'
gyraze_hd = pygkyl.GyrazeInterface(simulation)
gyraze_hd.load_h5_data(filename)
gyraze_hd.plot_data(alpha=1.0/25)


In [ ]:
fxy = lambda x: -0.05*x + 7
gyraze_hd.plot_attribute_scatter(attr_x='phi_norm0', attr_y='TioverTe', color_by='x0', fxy=None)
gyraze_hd.plot_attribute_scatter(attr_x='gamma0', attr_y='TioverTe', color_by='x0', fxy=None)